# Exercise 2 · Write the fixtures and tests first

You write the files and tests that fix what a reader must do.

**Goal:** Write three fixtures and their tests, and see each test catch a wrong reader.

**Why it matters:** You write these tests without Codex, so that the agent's reader and tests cannot share one mistake.

**Before you start:** the course setup.

**What the check measures:** the fixtures are right. Your tests pass on a right reader and catch each broken copy. You committed them.

**Done when:** the last cell prints `Done:`.

## The situation

`return_reader.py` holds `read_values`, with no body: exercise 3 writes it. `fixtures/good.xml` is a return. `test_return_reader.py` has one test. `broken/` holds three wrong readers. The files are made up.

## Step 1 · Write the practice files

Run the cell. It writes seven files into `practice/`, beside this notebook. It never overwrites a file.

Expect seven lines that start with `Wrote practice/`, or `Kept` for a file that is already there.

In [ ]:
# Step 1: write the practice files, keeping any that already exist
from pathlib import Path

READER = '''import xml.etree.ElementTree as ET

NS = {"efile": "http://www.irs.gov/efile"}
WAGES = "efile:ReturnData/efile:IRS1040/efile:WagesAmt"
NAMES = "efile:ReturnData/efile:IRS1040/efile:DependentDetail/efile:DependentFirstNm"


def read_values(xml_path: str) -> dict[str, str | list[str] | None]:
    """Read the wages and the dependents' names from one return.

    Args:
        xml_path: The return's XML file.

    Returns:
        "wages": WagesAmt's text, "" if empty, None if absent.
        "dependents": each DependentFirstNm's text, in document order.

    Raises:
        xml.etree.ElementTree.ParseError: The file is not well-formed XML.
    """
'''
BROKEN = "# A broken copy, for Step 6.\n" + READER
FILES = {
    "return_reader.py": READER + '    raise NotImplementedError("Exercise 3 writes this")\n',
    "fixtures/good.xml": """<!-- Every value in this file is made up. -->
<Return xmlns="http://www.irs.gov/efile">
  <ReturnHeader>
    <PrimarySSN>400001234</PrimarySSN>
  </ReturnHeader>
  <ReturnData>
    <IRS1040 documentId="DOC0001">
      <WagesAmt>41000</WagesAmt>
      <DependentDetail>
        <DependentFirstNm>TEST CHILD ONE</DependentFirstNm>
      </DependentDetail>
      <DependentDetail>
        <DependentFirstNm>TEST CHILD TWO</DependentFirstNm>
      </DependentDetail>
    </IRS1040>
  </ReturnData>
</Return>
""",
    "test_return_reader.py": """import xml.etree.ElementTree as ET

import pytest

from return_reader import read_values


@pytest.mark.parametrize("name, expected", [
    ("wages", "41000"),
    ("dependents", ["TEST CHILD ONE", "TEST CHILD TWO"]),
], ids=["wages", "dependents"])
def test_good_return(name: str, expected: str | list[str]) -> None:
    assert read_values("fixtures/good.xml")[name] == expected
""",
    "broken/zero_for_missing.py": BROKEN + '''    root = ET.parse(xml_path).getroot()
    return {"wages": root.findtext(WAGES, "0", NS), "dependents": [e.text for e in root.findall(NAMES, NS)]}
''',
    "broken/none_for_no_dependents.py": BROKEN + '''    root = ET.parse(xml_path).getroot()
    names = [e.text for e in root.findall(NAMES, NS)]
    return {"wages": root.findtext(WAGES, namespaces=NS), "dependents": names or None}
''',
    "broken/swallows_errors.py": BROKEN + '''    try:
        root = ET.parse(xml_path).getroot()
    except ET.ParseError:
        return {"wages": None, "dependents": []}
    return {"wages": root.findtext(WAGES, namespaces=NS), "dependents": [e.text for e in root.findall(NAMES, NS)]}
''',
    ".gitignore": "__pycache__/\n",
}

PRACTICE = Path("practice")
for name, text in FILES.items():
    path = PRACTICE / name
    if path.exists():
        print(f"Kept practice/{name}: it is already there.")
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding="utf-8")
        print(f"Wrote practice/{name}")

## Step 2 · Open a terminal and commit the start

Choose **Terminal > New Terminal**. Type each line, then press Enter:

```sh
source .venv/bin/activate
cd week-02/day-04/exercise-02-fixtures-and-tests-first/practice
git init -q -b main && git add -A && git commit -qm "Start"
```

Expect no output from the last line.

### Learn: one fixture per case

A fixture is a small file that a test reads. Today's deck, 4.2: "Fixtures first: each one fixes what the reader must return."

## Step 3 · Make the three fixtures

Each line copies `fixtures/good.xml` with one change. `grep -v` drops each line that holds the word. `head -n 10` keeps the first 10 of its 17 lines.

```sh
grep -v Dependent fixtures/good.xml > fixtures/no_dependents.xml
grep -v WagesAmt fixtures/good.xml > fixtures/no_wages.xml
head -n 10 fixtures/good.xml > fixtures/cut_off.xml
tail -n 1 fixtures/cut_off.xml
```

`tail -n 1` prints the last line. Expect the first `DependentFirstNm`. The file stops there, so its elements never close.

## Step 4 · Write three tests

Open the file with `code test_return_reader.py`. Below `test_good_return`, add one plain function per row of this table. Then save:

| Test name | Fixture | It expects |
| --- | --- | --- |
| `test_no_dependents_is_empty_list` | `fixtures/no_dependents.xml` | `["dependents"] == []` |
| `test_missing_wages_is_absent` | `fixtures/no_wages.xml` | `["wages"] is None` |
| `test_cut_off_file_is_an_error` | `fixtures/cut_off.xml` | `with pytest.raises(ET.ParseError):` around the call |

<details><summary>Show the answer</summary>

```python
def test_no_dependents_is_empty_list() -> None:
    assert read_values("fixtures/no_dependents.xml")["dependents"] == []


def test_missing_wages_is_absent() -> None:
    assert read_values("fixtures/no_wages.xml")["wages"] is None


def test_cut_off_file_is_an_error() -> None:
    with pytest.raises(ET.ParseError):
        read_values("fixtures/cut_off.xml")
```

Why:

- No dependents is still a list (4.2, "Test for absence with is None: a present wage tests False.": "findall: a list, in document order").
- No `WagesAmt` is `None`, not `"0"`: nobody reported a wage (4.2: "None when missing").
- `pytest.raises` passes only if the code inside it raises that error (4.2: "A cut-off file raises ParseError: let it reach the caller").

</details>

## Step 5 · Run the tests, and watch each fail

```sh
python -m pytest -q
```

Expect `5 failed` (`parametrize` runs the given test once per row), each on `NotImplementedError`. That is right: the reader does not exist yet.

## Step 6 · Test your tests on three wrong readers

Each line puts a broken copy in place of `return_reader.py`, runs the tests, and puts the empty one back:

```sh
cp broken/zero_for_missing.py return_reader.py; python -m pytest -q; git restore return_reader.py
cp broken/none_for_no_dependents.py return_reader.py; python -m pytest -q; git restore return_reader.py
cp broken/swallows_errors.py return_reader.py; python -m pytest -q; git restore return_reader.py
```

Expect `1 failed, 4 passed` three times. Each time, a different test fails.

## Step 7 · Commit the fixtures and tests

```sh
git add fixtures test_return_reader.py
git commit -qm "Add fixtures and tests"
```

Expect no output.

## Start over (only if you need it)

Set `START_OVER = True` and run the cell. It writes the Step 1 files again, with one test. Then set it back to `False` and go to Step 3.

Expect `Nothing changed.` while it is `False`.

In [ ]:
# Start over: only if START_OVER is True, write the Step 1 files again
START_OVER = False

if not START_OVER:
    print("Nothing changed.")
elif "FILES" not in globals():
    print("Run the Step 1 cell first, then this cell.")
else:
    for name, text in FILES.items():
        (PRACTICE / name).parent.mkdir(parents=True, exist_ok=True)
        (PRACTICE / name).write_text(text, encoding="utf-8")
    print("Wrote the practice files again. Now set START_OVER back to False.")

## Check your work

Run the cell. Expect four `ok:` lines, then `Done:`. A `Not yet:` line names the first problem and what to do.

In [ ]:
# Check your work: run your tests on a right reader and on each broken copy
import shutil
import subprocess
import sys
import tempfile
import xml.etree.ElementTree as ET
from pathlib import Path

PRACTICE = Path("practice")


def failed_tests(reader):
    """Return {test name: True if it failed}, with reader as return_reader.py."""
    with tempfile.TemporaryDirectory() as tmp:
        work = shutil.copytree(PRACTICE, Path(tmp) / "p", ignore=shutil.ignore_patterns(".git", "__pycache__"))
        (work / "return_reader.py").write_text(reader, encoding="utf-8")
        subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", "test_return_reader.py",
                        "--junitxml=r.xml"], cwd=work, capture_output=True, timeout=300)
        return {case.get("name"): any(case.find(tag) is not None for tag in ("failure", "error"))
                for case in ET.parse(work / "r.xml").iter("testcase")}


def tree(name):
    try:
        return ET.parse(PRACTICE / "fixtures" / name).getroot()
    except (OSError, ET.ParseError):
        return None


def first_problem():
    """Print ok for each check that passes; return the first problem, or ""."""
    for name, tag in (("no_dependents.xml", "DependentDetail"), ("no_wages.xml", "WagesAmt")):
        root = tree(name)
        if root is None or root.find(f".//{{http://www.irs.gov/efile}}{tag}") is not None:
            return f"fixtures/{name} is missing, does not parse, or still has {tag}. Make it (Step 3)."
    if not (PRACTICE / "fixtures" / "cut_off.xml").is_file() or tree("cut_off.xml") is not None:
        return "fixtures/cut_off.xml is missing, or it parses. Make it (Step 3)."
    print("ok: the three fixtures are right")
    right = (PRACTICE / "broken/zero_for_missing.py").read_text(encoding="utf-8")
    good = failed_tests(right.replace('WAGES, "0", NS', "WAGES, namespaces=NS"))  # the copy without its fault
    if len(good) < 5:
        return f"pytest runs {len(good)} tests, not 5 (the given test runs twice). Write three tests (Step 4)."
    wrong = [name for name, fails in good.items() if fails]
    if wrong:
        return f"{wrong[0]} fails on a right reader. Compare it with the answer (Step 4)."
    print("ok: every test passes on a right reader")
    for path in sorted((PRACTICE / "broken").glob("*.py")):
        if not any(failed_tests(path.read_text(encoding="utf-8")).values()):
            return f"no test fails on broken/{path.name}. Write the test that catches it (Step 4)."
    print("ok: each broken copy fails a test")
    if not (PRACTICE / ".git").exists():
        return "practice/ has no Git repository. Do Step 2."
    git = lambda *args: subprocess.run(["git", "-C", str(PRACTICE), *args], capture_output=True, text=True).stdout
    if len(git("log", "--format=%s").splitlines()) < 2 or git("status", "--porcelain", "fixtures", "test_return_reader.py"):
        return "the fixtures and tests are not committed. Commit them (Step 7)."
    print("ok: the fixtures and tests are committed")
    return ""


try:
    problem = first_problem()
except Exception as error:
    problem = f"the check stopped on {type(error).__name__}: {error}. Run Step 1, then python -m pytest -q (Step 5)."
print(f"Not yet: {problem}" if problem else "Done: each fixture fixes a case, and your tests catch each broken reader.")

## What you learned

- Each fixture fixes one case: absent, repeated or broken.
- A test counts once you see it catch a wrong reader.

**Reflect:** Which input in your own work has no fixture yet?